# 02 · The servable model

**Train/serve skew:** P1's model uses 48 features, but the live station feed only reliably provides PM2.5, PM10 and O₃ (the gas sensors have mislabeled units, and NH₃/benzene/toluene aren't reported).
Here we retrain on P1's data using **only features the product can fetch live**, and measure the cost against P1's full model.
Same data, same walk-forward monthly retraining, same test period (Jul–Dec 2025) as P1, so the numbers are directly comparable.

## 1. Data and feature sets

In [1]:
# Cell 3: Load P1's features, add next-day weather (as in P1), and define the feature sets.
from pathlib import Path                                 # File paths
import numpy as np                                       # Maths
import pandas as pd                                      # Tables

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data" / "features.csv", index_col="time", parse_dates=True)   # P1's hourly features

wx = pd.read_csv(ROOT / "data" / "weather.csv", skiprows=3, index_col="time", parse_dates=True).rename(columns={
    "temperature_2m (°C)": "temp", "relative_humidity_2m (%)": "humidity", "dew_point_2m (°C)": "dew_point",
    "precipitation (mm)": "rain", "wind_speed_10m (km/h)": "wind_speed", "wind_direction_10m (°)": "wind_dir",
    "surface_pressure (hPa)": "pressure", "cloud_cover (%)": "cloud"}).asfreq("h")   # P1's raw weather file
rad = np.deg2rad(wx["wind_dir"])
fut = pd.DataFrame(index=wx.index)                       # The weather 24 hours later (in live use: the forecast)
for col in ["temp", "humidity", "dew_point", "wind_speed", "pressure", "cloud"]:
    fut[f"{col}_next24"] = wx[col].shift(-24)
fut["wind_u_next24"] = (wx["wind_speed"] * np.sin(rad)).shift(-24)
fut["wind_v_next24"] = (wx["wind_speed"] * np.cos(rad)).shift(-24)
fut["rain_sum_next24"] = wx["rain"].rolling(24).sum().shift(-24)
df = df.join(fut).dropna(subset=list(fut.columns))       # Exactly as in P1

TARGET = "target_pm25_24h"
FULL = [c for c in df.columns if c not in [TARGET, "blh"]]   # P1's full feature set
GASES = ["no", "no2", "nox", "nh3", "so2", "co", "benzene", "toluene"]   # Not reliably available live
CORE = [c for c in FULL if c not in GASES + ["pm10", "o3"]]   # PM2.5 history + time + weather
FEATURE_SETS = {
    "P1 full (48 features, not servable)": FULL,
    "Servable: PM2.5 + time + weather": CORE,
    "Servable: + PM10": CORE + ["pm10"],
    "Servable: + PM10 + O3": CORE + ["pm10", "o3"],
}
for name, cols in FEATURE_SETS.items():
    print(f"{name:40s} {len(cols)} features")
print("Rows:", len(df), "|", df.index.min().date(), "->", df.index.max().date())

P1 full (48 features, not servable)      48 features
Servable: PM2.5 + time + weather         38 features
Servable: + PM10                         39 features
Servable: + PM10 + O3                    40 features
Rows: 15304 | 2024-01-03 -> 2025-12-30


## 2. Walk-forward evaluation (P1's method)
Each test month is predicted by a model retrained on everything known before it. All models are scored on exactly the same hours.

In [2]:
# Cell 5: Walk-forward evaluation of every feature set, plus the two baselines, on the same test hours.
from sklearn.ensemble import HistGradientBoostingRegressor   # P1's model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def make_model():
    return HistGradientBoostingRegressor(loss="absolute_error", max_iter=500, learning_rate=0.05,
                                         max_leaf_nodes=31, random_state=42)   # Exactly P1's settings


def walk_forward(cols):
    preds = []
    for month_start in pd.date_range("2025-07-01", "2025-12-01", freq="MS"):
        month_end = month_start + pd.offsets.MonthBegin(1)
        tr = df[df.index < month_start - pd.Timedelta(hours=24)]   # Only rows whose answer was known before this month
        te = df[(df.index >= month_start) & (df.index < month_end)]
        preds.append(pd.Series(make_model().fit(tr[cols], tr[TARGET]).predict(te[cols]), index=te.index))
    return pd.concat(preds)


def category(v):                                         # India's National AQI categories for PM2.5
    return "Good" if v <= 30 else "Satisfactory" if v <= 60 else "Moderate" if v <= 90 else \
           "Poor" if v <= 120 else "Very Poor" if v <= 250 else "Severe"


rows, predictions = [], {}
for name, cols in FEATURE_SETS.items():
    predictions[name] = walk_forward(cols)
idx = predictions[next(iter(predictions))].index         # The common test hours
y = df.loc[idx, TARGET]
predictions["Baseline: same as now"] = df.loc[idx, "pm25"]
predictions["Baseline: last 24h average"] = df.loc[idx, "pm25_mean_24h"]

for name, p in predictions.items():
    p = p.loc[idx]
    rows.append({"Model": name, "MAE": round(mean_absolute_error(y, p), 1),
                 "RMSE": round(float(np.sqrt(mean_squared_error(y, p))), 1), "R²": round(r2_score(y, p), 3),
                 "Category accuracy": f"{(p.apply(category) == y.apply(category)).mean():.1%}"})
results = pd.DataFrame(rows)
(ROOT / "reports").mkdir(exist_ok=True)
results.to_csv(ROOT / "reports" / "evaluation.csv", index=False)   # For the README
results

,Model,MAE,RMSE,R²,Category accuracy
0,"P1 full (48 features, not servable)",24.3,44.1,0.568,58.1%
1,Servable: PM2.5 + time + weather,24.1,43.7,0.575,59.9%
2,Servable: + PM10,24.2,43.9,0.571,60.1%
3,Servable: + PM10 + O3,24.2,44.1,0.568,60.0%
4,Baseline: same as now,23.4,49.7,0.451,59.0%
5,Baseline: last 24h average,23.9,46.2,0.525,59.7%


## 3. Save the chosen servable model
Trained on all data (as P1 did for its final model). Saved together with its feature list and metadata the API needs.

In [4]:
# Cell 7: Train the chosen feature set on all rows and save it with its metadata.
import joblib                                            # Save/load the model

CHOSEN = "Servable: PM2.5 + time + weather"              # Best result with the fewest live inputs
cols = FEATURE_SETS[CHOSEN]
final = make_model().fit(df[cols], df[TARGET])           # Learn from every row whose answer is known
(ROOT / "models").mkdir(exist_ok=True)
joblib.dump({"model": final, "features": cols, "feature_set": CHOSEN,
             "trained_until": str(df.index.max()), "station": "Shadipur, Delhi - CPCB (OpenAQ location 5630)",
             "test_metrics": results.set_index("Model").loc[CHOSEN].to_dict()},
            ROOT / "models" / "model.joblib")
print(f"Saved '{CHOSEN}' with {len(cols)} features, trained until {df.index.max()}")
print(cols)

Saved 'Servable: PM2.5 + time + weather' with 38 features, trained until 2025-12-30 23:00:00
['pm25', 'temp', 'humidity', 'dew_point', 'rain', 'wind_speed', 'pressure', 'cloud', 'hour', 'month', 'dayofweek', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_winter', 'wind_u', 'wind_v', 'pm25_lag_1h', 'pm25_lag_2h', 'pm25_lag_3h', 'pm25_lag_6h', 'pm25_lag_12h', 'pm25_lag_24h', 'pm25_lag_48h', 'pm25_mean_6h', 'pm25_mean_24h', 'pm25_std_24h', 'pm25_change_24h', 'temp_next24', 'humidity_next24', 'dew_point_next24', 'wind_speed_next24', 'pressure_next24', 'cloud_next24', 'wind_u_next24', 'wind_v_next24', 'rain_sum_next24']
